# Predict telco customer churn with KumoTabular

Which mobile and broadband customers are likely to cancel in the next 30 days, and who should a retention team contact first?

[KumoTabular](https://nvidia.github.io/structured-data-models/api/generated/sdm.models.KumoTabular.html) is a pretrained tabular foundation model. Instead of building, training and tuning a churn model, you give it labeled past customers as context and it scores new customers directly. Use it when you already have one row of features per customer and want a strong churn ranking quickly, for example as a proof of concept. To learn from raw related tables such as usage, bills and tickets without engineering features, see the [Kumo Engine churn notebook](../../../../kumo-engine/telco/notebooks/local_kumo_engine_synthetic_telco_churn.ipynb).

The workflow: supply labeled context rows, check ranking quality on held-out customers, score a new batch, explain the riskiest scores and save the context for reuse.

Requires a GPU and Python 3.11 or newer. The first run downloads the public [Kumo-Tabular weights](https://huggingface.co/nvidia/Kumo-Tabular). All data is synthetic, so metrics are **demonstration only**.

## Install

Install the Structured Data Models package and the data libraries used by this notebook.

In [1]:
%pip install -q structured-data-models pandas pyarrow scikit-learn

## Import libraries

In [2]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score

import sdm
from sdm.explain import GradientExplainer
from sdm.models import KumoTabular

## Set the example parameters

Each cohort is a monthly customer snapshot: March is the labeled context, April is held out for evaluation and June is the unlabeled batch to score. `TOP_SHARE` is the fraction of customers a retention team can contact. `MODEL_SIZE` can be raised to `"medium"` or `"large"`; keep `NUM_ESTIMATORS = 1` for the explanation step.

In [3]:
SEED = 7
N_CONTEXT, N_HELDOUT, N_SCORE = 3000, 1500, 1000
CONTEXT_TIME, HELDOUT_TIME, SCORE_TIME = "2025-03-01", "2025-04-15", "2025-06-01"
TOP_SHARE = 0.10

MODEL_SIZE = "small"
if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a GPU.")
DEVICE = "cuda"
NUM_ESTIMATORS = 1

run_dir = Path(tempfile.mkdtemp(prefix="telco-churn-tabular-"))
torch.manual_seed(SEED)
print("Outputs:", run_dir)

Outputs: /tmp/telco-churn-tabular-uw7crf8k


## Generate synthetic example data

Run the next cell to create one row per customer per snapshot: plan and contract status, billing, usage trend, network quality and support contacts, plus whether the customer cancelled within 30 days. You can skim the generation code; the Kumo workflow starts at **Prepare context, held-out and scoring rows**.

In [4]:
rng = np.random.default_rng(SEED)
cohort_sizes = {"context": N_CONTEXT, "heldout": N_HELDOUT, "score": N_SCORE}
snapshot_times = {"context": CONTEXT_TIME, "heldout": HELDOUT_TIME, "score": SCORE_TIME}
n = sum(cohort_sizes.values())
cohort = np.repeat(list(cohort_sizes), list(cohort_sizes.values()))

plan_prices = {"sim_only_rolling": 10, "sim_only_12m": 14, "handset_24m": 38, "broadband": 32, "broadband_tv": 68}
plan = rng.choice(list(plan_prices), n, p=[0.18, 0.2, 0.27, 0.2, 0.15])
region = rng.choice(["north", "south", "east", "west", "central"], n)
region_quality = pd.Series({"north": 1.6, "south": 0.8, "east": 1.0, "west": 1.2, "central": 0.7})[region].to_numpy()
num_lines = 1 + rng.poisson(0.5, n)
months_left = np.where(plan == "sim_only_rolling", 0, rng.integers(-18, 24, n))
out_of_contract = months_left <= 0
price_change = np.where(out_of_contract & (plan != "sim_only_rolling"), rng.choice([0, 15, 25], n), 0) + rng.choice([0, 3.9, 6.4], n)
dissatisfaction = rng.gamma(1.0, 0.6, n)
stress = rng.beta(1.5, 8, n)

customers = pd.DataFrame({
    "customer_id": [f"cust_{i:06d}" for i in range(n)],
    "cohort": cohort,
    "snapshot_date": pd.to_datetime([snapshot_times[c] for c in cohort]),
    "plan": plan,
    "region": region,
    "tenure_months": rng.integers(1, 120, n),
    "num_lines": num_lines,
    "monthly_charge": (np.array([plan_prices[p] for p in plan]) * (1 + 0.6 * (num_lines - 1)) * (1 + price_change / 100) * rng.lognormal(0, 0.08, n)).round(2),
    "contract_months_left": months_left,
    "price_change_pct_12m": price_change.round(1),
    "data_usage_gb_30d": (rng.gamma(2.0, 9.0, n) * np.exp(-0.3 * dissatisfaction)).round(1),
    "usage_change_pct_90d": (rng.normal(3, 10, n) - 14 * dissatisfaction).round(1),
    "dropped_calls_30d": rng.poisson(1.5 * region_quality * (1 + dissatisfaction)),
    "network_tickets_90d": rng.poisson(0.25 * region_quality + 0.6 * dissatisfaction),
    "billing_tickets_90d": rng.poisson(0.15 + 0.03 * price_change),
    "late_payments_6m": rng.poisson(4 * stress),
})

logit = (-3.3 + 1.1 * out_of_contract + 0.6 * (plan == "sim_only_rolling") - 0.6 * (plan == "broadband_tv")
         + 0.55 * np.minimum(customers.network_tickets_90d, 3) + 0.08 * customers.dropped_calls_30d
         + 0.05 * customers.price_change_pct_12m * out_of_contract + 0.35 * customers.billing_tickets_90d
         + 0.3 * customers.late_payments_6m - 0.03 * customers.usage_change_pct_90d
         - 0.3 * (customers.num_lines - 1) - 0.25 * np.log1p(customers.tenure_months) + rng.normal(0, 0.5, n))
churned = rng.binomial(1, 1 / (1 + np.exp(-logit)))
customers["churned_30d"] = np.where(cohort == "score", np.nan, churned)

print(customers.groupby("cohort", sort=False).churned_30d.agg(rows="size", churn_rate="mean"))
display(customers.head(3))

         rows  churn_rate
cohort                   
context  3000    0.148000
heldout  1500    0.146667
score    1000         NaN


,customer_id,cohort,snapshot_date,plan,region,tenure_months,num_lines,monthly_charge,contract_months_left,price_change_pct_12m,data_usage_gb_30d,usage_change_pct_90d,dropped_calls_30d,network_tickets_90d,billing_tickets_90d,late_payments_6m,churned_30d
0,cust_000000,context,2025-03-01,handset_24m,east,4,1,38.69,14,0.0,12.1,-5.8,3,1,0,0,0.0
1,cust_000001,context,2025-03-01,broadband_tv,east,65,2,121.03,21,6.4,12.0,-1.8,1,0,0,0,0.0
2,cust_000002,context,2025-03-01,broadband,west,36,2,63.03,-11,25.0,7.2,16.1,1,1,0,0,0.0


## Prepare context, held-out and scoring rows

There is no training step. The labeled March customers become the model's context, the April customers are held out to measure ranking quality, and the June customers are scored without labels. Only the feature columns reach the model; IDs, dates and cohort markers are excluded.

In [5]:
feature_columns = [
    "plan", "region", "tenure_months", "num_lines", "monthly_charge", "contract_months_left",
    "price_change_pct_12m", "data_usage_gb_30d", "usage_change_pct_90d", "dropped_calls_30d",
    "network_tickets_90d", "billing_tickets_90d", "late_payments_6m",
]
target = "churned_30d"

context = customers[customers.cohort == "context"]
heldout = customers[customers.cohort == "heldout"]
scoring = customers[customers.cohort == "score"].reset_index(drop=True)

context_df = context[feature_columns + [target]].astype({target: int}).reset_index(drop=True)
y_heldout = heldout[target].astype(int).to_numpy()
print(f"Context: {len(context_df)} | Held-out: {len(heldout)} | Scoring: {len(scoring)}")

Context: 3000 | Held-out: 1500 | Scoring: 1000


Convert the frames to SDM tensors. Infer column types from the context, and mark the 0/1 churn label as categorical so the model treats this as classification.

In [6]:
context_table = sdm.TableTensor.from_pandas(
    context_df, stypes=sdm.infer_stypes(context_df, overrides={target: "categorical"}), device=DEVICE
)
x_context = context_table.drop_columns(target)
y_context = context_table[:, target]
feature_stypes = sdm.infer_stypes(context_df[feature_columns])
x_heldout = sdm.TableTensor.from_pandas(heldout[feature_columns], stypes=feature_stypes, device=DEVICE)
x_score = sdm.TableTensor.from_pandas(scoring[feature_columns], stypes=feature_stypes, device=DEVICE)

## Load the pretrained model and fit the context

Load the small checkpoint ([OpenMDW license](https://huggingface.co/nvidia/Kumo-Tabular)). `fit` stores the context rows; it does not update model weights, so there is nothing to train or tune.

In [7]:
model = KumoTabular(task="classification", size=MODEL_SIZE, device=DEVICE)
model.fit(x=x_context, y=y_context, num_estimators=NUM_ESTIMATORS,
          generator=torch.Generator(device=DEVICE).manual_seed(SEED))

## Evaluate on held-out customers

AUROC measures how well the model ranks churners above non-churners. For a retention campaign, the more direct measure is how many churners fall in the top `TOP_SHARE` of scores. Here the top 10% have a churn rate about three times the overall rate, so the same outreach budget reaches about a third of next month's churners. **Synthetic-data metrics: demonstration only.**

In [8]:
def churn_probability(prediction):
    class_names = list(prediction.columns[sdm.Stype.numerical])
    return prediction.numerical.detach().cpu().numpy()[:, class_names.index("1")]


def top_share(probabilities):
    contact_count = round(TOP_SHARE * len(probabilities))
    return pd.Series(probabilities).rank(method="first", ascending=False).to_numpy() <= contact_count


p_heldout = churn_probability(model.predict(x_heldout))

top = top_share(p_heldout)
metrics = pd.Series({
    "AUROC": roc_auc_score(y_heldout, p_heldout),
    "Average precision": average_precision_score(y_heldout, p_heldout),
    "Overall churn rate": y_heldout.mean(),
    f"Churn rate in top {TOP_SHARE:.0%}": y_heldout[top].mean(),
    f"Share of churners in top {TOP_SHARE:.0%}": y_heldout[top].sum() / y_heldout.sum(),
}, name="held-out").round(3)
display(metrics.to_frame())

,held-out
AUROC,0.798
Average precision,0.456
Overall churn rate,0.147
Churn rate in top 10%,0.487
Share of churners in top 10%,0.332


## Batch predictions

Score the June snapshot with the same context. Each scoring row receives one churn probability.

In [9]:
p_score = churn_probability(model.predict(x_score))

### Inspect and save the scores

Attach the probabilities to customer IDs in input order and flag exactly the top `TOP_SHARE` of customers as a retention priority. The flag is an illustrative campaign cut-off, not a calibrated decision threshold.

In [10]:
predictions = pd.DataFrame({
    "customer_id": scoring.customer_id,
    "snapshot_date": scoring.snapshot_date,
    "churn_probability": p_score,
    "retention_priority": top_share(p_score),
})
assert len(predictions) == len(scoring)
predictions.to_parquet(run_dir / "predictions.parquet", index=False)
display(predictions.sort_values("churn_probability", ascending=False).head(5))
print(f"Saved {len(predictions)} predictions to", run_dir / "predictions.parquet")

,customer_id,snapshot_date,churn_probability,retention_priority
510,cust_005010,2025-06-01,0.866048,True
51,cust_004551,2025-06-01,0.810825,True
36,cust_004536,2025-06-01,0.803690,True
489,cust_004989,2025-06-01,0.801787,True
493,cust_004993,2025-06-01,0.774026,True


Saved 1000 predictions to /tmp/telco-churn-tabular-uw7crf8k/predictions.parquet


## Explain the highest-risk customers

[GradientExplainer](https://nvidia.github.io/structured-data-models/api/generated/sdm.explain.GradientExplainer.html) measures how sensitive each churn probability is to each input, including the categorical `plan` and `region`. For the riskiest customers here, late payments and billing contacts dominate, which points the retention team to billing-focused offers. Sensitivity is not a causal effect.

In [11]:
riskiest = np.argsort(-p_score)[:5]
attributions, _ = GradientExplainer().explain(model, x_score[riskiest])
churn_class = list(model.predict(x_score[riskiest]).columns[sdm.Stype.numerical]).index("1")
gradients = attributions.numerical.detach().cpu().numpy()[churn_class]
drivers = pd.DataFrame(np.abs(gradients), columns=list(attributions.columns[sdm.Stype.numerical]),
                       index=scoring.customer_id.iloc[riskiest])
display(drivers.mean().sort_values(ascending=False).rename("mean |gradient|").to_frame())

,mean |gradient|
late_payments_6m,1.454464
billing_tickets_90d,1.099765
network_tickets_90d,0.401423
num_lines,0.299152
tenure_months,0.271113
dropped_calls_30d,0.076381
data_usage_gb_30d,0.047753
usage_change_pct_90d,0.047037
monthly_charge,0.037219
price_change_pct_12m,0.036575


## Retain and reuse

KumoTabular has no trained weights to save; the reusable artifact is the context. Save the labeled context, then rebuild a fresh model from the file and confirm it reproduces the June scores.

In [12]:
context_df.to_parquet(run_dir / "context.parquet", index=False)

restored_context = pd.read_parquet(run_dir / "context.parquet")
restored_table = sdm.TableTensor.from_pandas(
    restored_context, stypes=sdm.infer_stypes(restored_context, overrides={target: "categorical"}), device=DEVICE
)
restored_model = KumoTabular(task="classification", size=MODEL_SIZE, device=DEVICE)
restored_model.fit(x=restored_table.drop_columns(target), y=restored_table[:, target],
                   num_estimators=NUM_ESTIMATORS, generator=torch.Generator(device=DEVICE).manual_seed(SEED))
assert np.allclose(churn_probability(restored_model.predict(x_score)), p_score, atol=1e-5)
print("Rebuilt model reproduces the saved predictions.")

Rebuilt model reproduces the saved predictions.


## Cleanup

Release the context held by both models. The saved context and predictions remain in `run_dir`.

In [13]:
model.clear()
restored_model.clear()
print("Outputs retained in", run_dir)

Outputs retained in /tmp/telco-churn-tabular-uw7crf8k


## Use your own data

Replace the generation cell with a `customers` DataFrame of your own snapshots. It needs `customer_id`, `snapshot_date`, a `cohort` column with the values `"context"`, `"heldout"` and `"score"`, your feature columns and a 0/1 `churned_30d` label that is missing for `"score"` rows. Update `feature_columns`; the rest of the notebook runs as written. If your data lives in several related tables, use the [Kumo Engine churn notebook](../../../../kumo-engine/telco/notebooks/local_kumo_engine_synthetic_telco_churn.ipynb).